In [2]:
%cd ..

/mnt/data1tb/thangcn/datnv2


In [3]:
store = {}

In [4]:
from typing import List, Dict, Any
from langchain_core.runnables.history import RunnableWithMessageHistory
from prompts.prompt import contextualize_q_system_prompt, qa_system_prompt
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_openai import ChatOpenAI
from service.func_for_fc import rag_doctor_info, rag_service_info, rag_product_info
import os
import json
import time
import streamlit as st
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory 
import uuid

from sqlalchemy import create_engine, Column, Integer, String, Text, ForeignKey
from sqlalchemy.orm import sessionmaker, relationship, declarative_base
from sqlalchemy.exc import SQLAlchemyError



/home/duyhoang/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/data1tb/thangcn/datnv2/service/func_for_fc.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [5]:
DATABASE_URL = "sqlite:///chat_history.db"
Base = declarative_base()

class Session(Base):
    __tablename__ = "sessions"
    id = Column(Integer, primary_key=True)
    session_id = Column(String, unique=True, nullable=False)
    messages = relationship("Message", back_populates="session")

class Message(Base):
    __tablename__ = "messages"
    id = Column(Integer, primary_key=True)
    session_id = Column(Integer, ForeignKey("sessions.id"), nullable=False)
    role = Column(String, nullable=False)
    content = Column(Text, nullable=False)
    session = relationship("Session", back_populates="messages")

# Create the database and the tables
engine = create_engine(DATABASE_URL)
Base.metadata.create_all(engine)
SessionLocal = sessionmaker(bind=engine)

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

# Function to save a single message
def save_message(session_id: str, role: str, content: str):
    db = next(get_db())
    try:
        session = db.query(Session).filter(Session.session_id == session_id).first()
        if not session:
            session = Session(session_id=session_id)
            db.add(session)
            db.commit()
            db.refresh(session)

        db.add(Message(session_id=session.id, role=role, content=content))
        db.commit()
    except SQLAlchemyError:
        db.rollback()
    finally:
        db.close()

# Function to load chat history
def load_session_history(session_id: str) -> BaseChatMessageHistory:
    db = next(get_db())
    chat_history = ChatMessageHistory()
    try:
        session = db.query(Session).filter(Session.session_id == session_id).first()
        if session:
            for message in session.messages:
                chat_history.add_message({"role": message.role, "content": message.content})
    except SQLAlchemyError:
        pass
    finally:
        db.close()

    return chat_history
def clear_all_data():
    db = SessionLocal()
    try:
        # Xóa tất cả message trước
        db.query(Message).delete()
        # Sau đó xóa tất cả session
        db.query(Session).delete()
        db.commit()
        print("Đã xóa toàn bộ dữ liệu trong database")
    except Exception as e:
        db.rollback()
        print(f"Lỗi khi xóa dữ liệu: {e}")
    finally:
        db.close()

# Gọi hàm để xóa dữ liệu
# clear_all_data()

In [6]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = load_session_history(session_id)
    return store[session_id]

# Ensure you save the chat history to the database when needed
def save_all_sessions():
    for session_id, chat_history in store.items():
        for message in chat_history.messages:
            save_message(session_id, message["role"], message["content"])

import atexit
atexit.register(save_all_sessions)

load_dotenv('/mnt/data1tb/thangcn/datnv2/.env')
# Lấy các khóa API và mô hình
open_ai_key = os.getenv("OPENAI_API_KEY")
MODEL = 'gpt-4o' #os.getenv("MODEL", "gpt-4o")
EMBED_MODEL = "nampham1106/bkcare-embedding" #os.getenv("EMBED_MODEL", "nampham1106/bkcare-embedding")

with open('/mnt/data1tb/thangcn/datnv2/prompts/tools.json', 'r') as f:
    function_schema = json.load(f)

In [6]:
session_id = str(uuid.uuid4())

In [7]:
session_id = 'thangcn1943'

In [8]:
clear_all_data()

Đã xóa toàn bộ dữ liệu trong database


In [10]:
chat_history = load_session_history(session_id)

In [11]:
chat_history.messages

[{'role': 'user',
  'content': 'tôi đang quan tâm đến khoa Da liễu, bạn có thể cung cấp cho tôi thông tin các bác sĩ được không'},
 {'role': 'assistant',
  'content': 'Các bác sĩ tại khoa Da liễu bao gồm: BSCK I. Trần Đức Anh, ThS.BS Đỗ Thị Thúy Hồng, ThS.BSCKII. Trần Thái Sơn (Trưởng khoa Da liễu), BS. Phan Nữ Thục Hiền, ThS. BS. Nguyễn Ngọc Oanh, ThS.BSNT. Nguyễn Thị Huế, ThS.BSNT. Vũ Duy Linh, ThS.BSNT. Nguyễn Thị Thu Phương, và BSCKII. Dương Thị Hằng (Phó trưởng khoa Da liễu - Bv Bạch Mai).'},
 {'role': 'user', 'content': 'Nước Colgate Plax có giá bao nhiêu'},
 {'role': 'assistant',
  'content': 'Tôi không có thông tin về giá của nước súc miệng Colgate Plax trong dữ liệu hiện tại.'},
 {'role': 'user',
  'content': 'Nước súc miệng  Oral-B Mouthwash có giá bao nhiêu'},
 {'role': 'assistant',
  'content': 'Tôi không có thông tin về giá của nước súc miệng Oral-B Mouthwash trong dữ liệu hiện tại.'},
 {'role': 'user',
  'content': 'Nước súc miệng Listerine Cool Mint có giá bao nhiêu'},
 

In [6]:
from dotenv import load_dotenv
from sentence_transformers import CrossEncoder
from langchain.retrievers import EnsembleRetriever  
from langchain_community.retrievers import BM25Retriever  
from langchain_core.documents import Document  

In [ ]:
class ReRankerRetriever(EnsembleRetriever):
    def __init__(self, ensemble_retriever, reranker, top_k=20, rerank_k=10):
        self.ensemble_retriever = ensemble_retriever,
        self.reranker = reranker
        self.top_k = top_k
        self.rerank_k = rerank_k 
    
    def _get_relevant_documents(self, query: str):
        docs = self.ensemble_retriever.get_relevant_documents(query)
        
        pairs = [(query, doc.page_content) for doc in docs]
   
        rerank_scores = self.reranker.predict(pairs)
 
        for doc, score in zip(docs, rerank_scores):
            doc.metadata["rerank_score"] = score
    
        sorted_docs = sorted(docs, key=lambda x: x.metadata["rerank_score"], reverse=True)
        return sorted_docs[:self.rerank_k]

In [9]:
llm = ChatOpenAI(model=MODEL, temperature=0, api_key=open_ai_key)
def process_llm_function_call(chat_history, user_prompt: str):
    messages = []
    for msg in chat_history.messages:
        messages.append(msg)
    # Thêm câu hỏi mới nhất
    messages.append(
        {"role": "user", "content": user_prompt}
    ) 
    # Gọi LLM với function calling
    response = llm.predict_messages(
        messages,
        functions=function_schema
    )
    print(response)
    return response

In [14]:
user_prompt = "Kể tên các bác sĩ khác trong khoa"
r = process_llm_function_call(load_session_history(session_id), user_prompt)

# Kiểm tra xem có function_call trong response không
if 'function_call' in r.additional_kwargs:
    available_functions = {tool['name']: globals()[tool['name']] 
                         for tool in function_schema}
    function_args = json.loads(r.additional_kwargs['function_call']['arguments'])
    function_name = r.additional_kwargs['function_call']['name']
    query = function_args['query']
    print(function_name)
    function_to_call = available_functions.get(function_name)
    function_response = function_to_call(**function_args)
else:
    # Xử lý trường hợp không có function call
    print("Không có function call trong response")
    function_response = r.content

content='Bạn có thể cho tôi biết tên khoa hoặc chuyên khoa mà bạn đang quan tâm để tôi có thể cung cấp thông tin về các bác sĩ trong khoa đó không?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 157, 'total_tokens': 192, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_6dd05565ef', 'finish_reason': 'stop', 'logprobs': None} id='run-ee226695-850c-42b1-bdd8-743099ba81ca-0' usage_metadata={'input_tokens': 157, 'output_tokens': 35, 'total_tokens': 192, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Không có function call trong response


In [13]:
r.content

'Bạn có thể cho tôi biết tên khoa hoặc chuyên khoa mà bạn muốn tìm hiểu về các bác sĩ không?'

In [ ]:
user_prompt = "Kể tên các bác sĩ khác trong khoa"
r = process_llm_function_call(load_session_history(session_id), user_prompt)

# Kiểm tra xem có function_call trong response không
if 'function_call' in r.additional_kwargs:
    available_functions = {tool['name']: globals()[tool['name']] 
                         for tool in function_schema}
    function_args = json.loads(r.additional_kwargs['function_call']['arguments'])
    function_name = r.additional_kwargs['function_call']['name']
    function_to_call = available_functions.get(function_name)
    function_response = function_to_call(**function_args)
else:
    # Xử lý trường hợp không có function call
    print("Không có function call trong response")
    function_response = None  # hoặc xử lý theo cách khác tùy yêu cầu

# Prompt để ngữ cảnh hóa câu hỏi
contextualize_q_system_prompt = """Given a chat history and the latest user question \
which might reference context in the chat history, formulate a standalone question \
which can be understood without the chat history. Do NOT answer the question, \
Do NOT repeat what has already been answered.\
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [("system", contextualize_q_system_prompt), MessagesPlaceholder("chat_history"), ("human", "{input}")]
)

# Kiểm tra function_response trước khi sử dụng
if function_response is not None:
    cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    rerank_retriever = ReRankerRetriever(function_response, cross_encoder,20,10)
    history_aware_retriever = create_history_aware_retriever(llm, rerank_retriever, contextualize_q_prompt)
else:
    # Xử lý trường hợp không có function response
    history_aware_retriever = None  # hoặc tạo một retriever mặc định

# Prompt để trả lời câu hỏi
qa_system_prompt = """You are an assistant for question-answering tasks. \
Use the following pieces of retrieved context to answer the question. \
If you don't know the answer, just say that you don't know. \
Use three sentences maximum and keep the answer concise.\n\n{context}"""

qa_prompt = ChatPromptTemplate.from_messages(
    [("system", qa_system_prompt), MessagesPlaceholder("chat_history"), ("human", "{input}")]
)
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

if history_aware_retriever is not None:
    rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)
else:
    # Xử lý trường hợp không có retriever
    rag_chain = question_answer_chain  # hoặc xử lý khác tùy yêu cầu

store = {}
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

history_conversation = f"""
    History conversation:\n
    """
for msg in load_session_history(session_id).messages[max(-len(load_session_history(session_id).messages), -5):]:
    history_conversation += f"'role' : '{msg['role']}' , 'content' : ' {msg['content']}' \n"

print(history_conversation)

def invoke_and_save(session_id, input_text):

    save_message(session_id, "user", input_text)
    
    result = conversational_rag_chain.invoke(
        {"input": history_conversation + '\nQuery: ' + input_text},
        config={"configurable": {"session_id": session_id}}
    )["answer"]

    save_message(session_id, "assistant", result)
    return result

content='' additional_kwargs={'function_call': {'arguments': '{"query":"bác sĩ khoa Da Liễu"}', 'name': 'rag_doctor_info'}, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 194, 'total_tokens': 219, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_c1e1ac6736', 'finish_reason': 'function_call', 'logprobs': None} id='run-3c759034-406b-445b-9f24-6b6cbe55e4dd-0' usage_metadata={'input_tokens': 194, 'output_tokens': 25, 'total_tokens': 219, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
rag_doctor_info

    History conversation:

    'role' : 'user' , 'content' : ' Ai là trưởng khoa Da Liễu' 
'role' : 'assistant' , 'content' : ' Trưởng khoa Da liễu là ThS.BSCKII. Trần Thái

In [13]:
result = invoke_and_save(session_id, user_prompt)
print(result)

Các bác sĩ khác trong khoa Da liễu bao gồm: BSCK I. Trần Đức Anh, ThS. BS. Nguyễn Ngọc Oanh, BSCKII. Dương Thị Hằng, BS. Phan Nữ Thục Hiền, ThS.BS Đỗ Thị Thúy Hồng, ThS. BS. Hoàng Hồng Mạnh, ThS.BSNT. Vũ Duy Linh, TS.BS. Hoàng Thị Hoạt, ThS.BSNT. Nguyễn Thị Huế, và ThS.BSNT. Nguyễn Thị Thu Phương.


In [ ]:
user_prompt = "Cac bac si khac trong khoa la nhung ai?"

In [ ]:
print(invoke_and_save(session_id, user_prompt))

In [ ]:
r = process_llm_function_call(load_session_history(session_id), user_prompt)
answer = None
Is_call_function = False
# Kiểm tra xem có function_call trong response không
if 'function_call' in r.additional_kwargs:
    available_functions = {tool['name']: globals()[tool['name']] 
                        for tool in function_schema}
    function_args = json.loads(r.additional_kwargs['function_call']['arguments'])
    function_name = r.additional_kwargs['function_call']['name']
    query = function_args['query']
    print(function_name)
    function_to_call = available_functions.get(function_name)
    function_response = function_to_call(**function_args)
    Is_call_function = True
else:
    # Xử lý trường hợp không có function call
    print("Không có function call trong response")
    function_response = r.content
if Is_call_function:
# Prompt để ngữ cảnh hóa câu hỏi

    contextualize_q_prompt = ChatPromptTemplate.from_messages(
        [("system", contextualize_q_system_prompt), MessagesPlaceholder("chat_history"), ("human", "{input}")]
    )
    history_aware_retriever = create_history_aware_retriever(llm, function_response, contextualize_q_prompt)

    qa_prompt = ChatPromptTemplate.from_messages(
        [("system", qa_system_prompt), MessagesPlaceholder("chat_history"), ("human", "{input}")]
    )

    question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

    rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)


    store = {}
    def get_session_history(session_id: str) -> BaseChatMessageHistory:
        if session_id not in store:
            store[session_id] = ChatMessageHistory()
        return store[session_id]

    conversational_rag_chain = RunnableWithMessageHistory(
        rag_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="chat_history",
        output_messages_key="answer",
    )

    history_conversation = f"""
        History conversation:\n
        """
    for msg in load_session_history(session_id).messages[max(-len(load_session_history(session_id).messages), -5):]:
        history_conversation += f"'role' : '{msg['role']}' , 'content' : ' {msg['content']}' \n"

    def invoke_and_save(session_id, input_text):

        # Save the user question with role "human"
        save_message(session_id, "user", input_text)
        
        result = conversational_rag_chain.invoke(
            {"input": history_conversation + '\nQuery: ' + input_text},
            config={"configurable": {"session_id": session_id}}
        )["answer"]
        # Save the AI answer with role "ai"
        save_message(session_id, "assistant", result)
        return result

    answer = invoke_and_save(session_id, user_prompt)
else:
    answer = function_response